# Download data — chapter 1

This notebook downloads the source datasets used by `slavic-speech-pipeline` from the CLARIN.SI repository.

**What it does**

- Download one or more datasets into `data/raw/<dataset>/`, sequentially.
- Unpack archives into `data/unpacked/<dataset>/`.
- Idempotent: skips files that already exist and are non-empty.
- Refuses to download multi-GB files unless `confirm_large=True`.

**What it does *not* do**

- Convert anything into canonical JSONL — that's `prep_<dataset>.ipynb`'s job.
- Cut WAVs — that's `audio_splitter.ipynb`'s job.
- Manage CLARIN auth for restricted resources (e.g. GOS audio). Those need a manual step.

---

## 0. Imports and project root setup

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Chapter dir  = {HERE}")

---

## 1. Config

Set `datasets` to what you want to download, then Run All.

- Single dataset: `["ROG"]`
- Multiple: `["ROG", "ParlaSpeech-HR"]`
- All ParlaSpeech languages at once: `["ParlaSpeech"]`

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # Which dataset(s) to fetch.
    # Registry keys: "ROG-Dialog", "ROG", "GOS",
    #                "ParlaSpeech-HR", "ParlaSpeech-RS", "ParlaSpeech-PL", "ParlaSpeech-CZ"
    # Shorthand:     "ParlaSpeech"  →  all four PS languages
    datasets: list = field(default_factory=lambda: ["ROG-Dialog"])

    # Allow files marked is_large=True to download (safety switch).
    confirm_large: bool = True                                         ############ Download Safety Switch

    # Force re-download even if file already exists.
    force: bool = False

    # Skip the unpack step.
    download_only: bool = False

    # Test mode: plan without fetching.
    test_mode: bool = False

cfg = Config()
print(cfg)

---

## 2. Dataset registry

Single source of truth for what can be downloaded. Sizes are approximate.

`"ParlaSpeech"` expands to all four PS language keys (HR → RS → PL → CZ).

In [ ]:
DATASETS = {
    # ── Slovenian ────────────────────────────────────────────────────────────
    "ROG-Dialog": {
        "handle":   "http://hdl.handle.net/11356/2073",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2073",
        "files": [
            # (filename, approx_size_mb, is_large)
            ("ROG-Dialog.zip",       5,    True),
            ("ROG-Dialog_audio.zip", 1220, True),
        ],
        "notes": (
            "Dialogue corpus — sentiment, dialogue-act, filled-pause annotations. "
            "25 speakers, 5.2 h, EXB/TRS/TXT. "
            "ROG-Dialog_audio.zip (~1.2 GB) required for audio tasks."
        ),
    },
    "ROG": {
        "handle":   "http://hdl.handle.net/11356/2062",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2062",
        "files": [
            ("ROG.zip",         30,   False),
            ("ROG-Art.wav.zip", 1400, False),
        ],
        "notes": "Read-speech + arts subcorpora. ROG-Art.wav.zip (~1.4 GB) required for audio.",
    },
    "GOS": {
        "handle":   "http://hdl.handle.net/11356/1863",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1863",
        "files": [
            ("Gos.TEI.zip",  60, False),
            ("Gos.TRS.zip",  50, False),
            ("Gos.TXT.zip",  10, False),
            ("Gos.vert.zip", 30, False),
        ],
        "notes": (
            "Spontaneous speech. Audio lives on a separate restricted handle "
            "(http://hdl.handle.net/11356/1973) — request access manually."
        ),
    },
    # ── ParlaSpeech ──────────────────────────────────────────────────────────
    "ParlaSpeech-HR": {
        "handle":   "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-HR.v3.0.jsonl.gz",     800,  False),
            ("ParlaSpeech-HR.v3.0.vert.gz",      400,  False),
            ("ParlaSpeech-HR.v3.0.textgrid.tgz", 1500, True),
        ],
        "notes": "Croatian parliamentary speech. Audio not bundled — see ParlaSpeech 2.0 for HR audio.",
    },
    "ParlaSpeech-RS": {
        "handle":   "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-RS.v3.0.jsonl.gz",     300, False),
            ("ParlaSpeech-RS.v3.0.vert.gz",      150, False),
            ("ParlaSpeech-RS.v3.0.textgrid.tgz", 500, True),
        ],
        "notes": "Serbian parliamentary speech. Audio not bundled.",
    },
    "ParlaSpeech-PL": {
        "handle":   "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-PL.v3.0.jsonl.gz", 500, False),
            ("ParlaSpeech-PL.v3.0.vert.gz",  250, False),
        ],
        "notes": "Polish parliamentary speech. No TextGrid in v3.0. Audio not bundled.",
    },
    "ParlaSpeech-CZ": {
        "handle":   "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-CZ.v3.0.jsonl.gz", 600, False),
            ("ParlaSpeech-CZ.v3.0.vert.gz",  300, False),
        ],
        "notes": "Czech parliamentary speech. No TextGrid in v3.0. Audio not bundled.",
    },
}

# ── ParlaSpeech shorthand ─────────────────────────────────────────────────────
PARLASPEECH_LANGS = ["ParlaSpeech-HR", "ParlaSpeech-RS", "ParlaSpeech-PL", "ParlaSpeech-CZ"]

def resolve_datasets(requested):
    """Expand 'ParlaSpeech' shorthand → all four PS language keys. Deduplicates."""
    out = []
    for name in requested:
        if name == "ParlaSpeech":
            out.extend(PARLASPEECH_LANGS)
        else:
            out.append(name)
    seen = set()
    return [x for x in out if not (x in seen or seen.add(x))]

target_datasets = resolve_datasets(cfg.datasets)
print(f"Available: {list(DATASETS.keys())}")
print(f"Requested: {cfg.datasets}")
print(f"Resolved:  {target_datasets}")

for name in target_datasets:
    if name not in DATASETS:
        raise ValueError(f"Unknown dataset {name!r}. Choose from {list(DATASETS)} or 'ParlaSpeech'.")

---

## 3. Plan the download

Lists every file per dataset, marks what will be skipped (already present, too large, test mode).
Nothing touches disk yet.

In [ ]:
plan = []  # flat list; each entry tracks its parent dataset

for ds_name in target_datasets:
    spec    = DATASETS[ds_name]
    raw_dir = PROJECT_ROOT / "data" / "raw" / ds_name
    raw_dir.mkdir(parents=True, exist_ok=True)

    udp.banner(f"Download plan — {ds_name}", char="-")
    print(f"notes: {spec['notes']}\n")

    for fname, size_mb, is_large in spec["files"]:
        dest    = raw_dir / fname
        url     = f"{spec['base_url']}/{fname}"
        already = dest.exists() and dest.stat().st_size > 0

        if already and not cfg.force:
            action, reason = "skip", "already downloaded"
        elif is_large and not cfg.confirm_large:
            action, reason = "skip", f"large ({size_mb} MB) — set confirm_large=True"
        elif cfg.test_mode:
            action, reason = "skip", "test_mode"
        else:
            action, reason = "download", "ok"

        plan.append({
            "dataset":  ds_name,
            "raw_dir":  raw_dir,
            "filename": fname,
            "url":      url,
            "dest":     dest,
            "size_mb":  size_mb,
            "is_large": is_large,
            "action":   action,
            "reason":   reason,
        })

        flag = "📥" if action == "download" else "⏭️ "
        print(f"  {flag} {fname:55s} ~{size_mb:>5} MB   [{action}: {reason}]")

    ds_total = sum(p["size_mb"] for p in plan if p["dataset"] == ds_name and p["action"] == "download")
    print(f"\n  → ~{ds_total} MB into {raw_dir}\n")

total_dl = sum(p["size_mb"] for p in plan if p["action"] == "download")
print(f"Grand total to download: ~{total_dl} MB")

---

## 4. Download helper

Streams to a `.part` file, renames on success — aborted download never leaves a half-finished file.

In [ ]:
import requests
from tqdm.auto import tqdm

def download_file(url: str, dest, *, chunk_size: int = 1 << 20):
    """Download url to dest. Streams via .part file for atomicity."""
    from pathlib import Path
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with part.open("wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, unit_divisor=1024,
            desc=dest.name, leave=True,
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    part.rename(dest)
    return dest

---

## 5. Execute the plan

Loop over `plan`, download everything marked `action == "download"`.

In [ ]:
if cfg.test_mode:
    print("🧪 TEST MODE: skipping all downloads")
else:
    for p in plan:
        if p["action"] != "download":
            print(f"⏭️  [{p['dataset']}] {p['filename']}  ({p['reason']})")
            continue
        print(f"📥 [{p['dataset']}] {p['filename']}  ←  {p['url']}")
        try:
            download_file(p["url"], p["dest"])
            print(f"   ✅ wrote {p['dest']}  ({p['dest'].stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"   ❌ failed: {e}")

print("\nDone.")

---

## 6. Unpack archives

`.zip` → unzip, `.tgz`/`.tar.gz` → tar, `.jsonl.gz`/`.gz` → single-file decompress.

Idempotent: if the unpacked directory exists and is non-empty, skip.

In [ ]:
import zipfile, tarfile, gzip, shutil

def unpack(archive, dest_dir) -> None:
    from pathlib import Path
    archive  = Path(archive)
    dest_dir = Path(dest_dir)
    name = archive.name
    stem = name
    for suf in (".tar.gz", ".tgz", ".jsonl.gz", ".zip", ".gz"):
        if name.endswith(suf):
            stem = name[: -len(suf)]
            break
    out = dest_dir / stem

    if out.exists() and any(out.iterdir()):
        print(f"  ⏭️  {name}  already unpacked → {out.relative_to(PROJECT_ROOT)}")
        return

    out.mkdir(parents=True, exist_ok=True)
    print(f"  📦 unpacking {name} → {out.relative_to(PROJECT_ROOT)}")

    if name.endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(out)
    elif name.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive, "r:gz") as tf:
            tf.extractall(out)
    elif name.endswith(".jsonl.gz") or name.endswith(".gz"):
        decomp_name = name[: -len(".gz")]
        with gzip.open(archive, "rb") as f_in, (out / decomp_name).open("wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    else:
        print(f"     ⚠️  no unpacker for {name}, skipping")
        return

    print(f"     ✅ unpacked")


if cfg.download_only:
    print("⏭️  download_only=True — skipping unpack")
elif cfg.test_mode:
    print("🧪 TEST MODE: skipping unpack")
else:
    for ds_name in target_datasets:
        udp.banner(f"Unpacking — {ds_name}", char="-")
        unpacked_dir = PROJECT_ROOT / "data" / "unpacked" / ds_name
        unpacked_dir.mkdir(parents=True, exist_ok=True)
        for p in [x for x in plan if x["dataset"] == ds_name]:
            if not p["dest"].exists():
                print(f"  ⏭️  {p['filename']}  (not on disk, can't unpack)")
                continue
            try:
                unpack(p["dest"], unpacked_dir)
            except Exception as e:
                print(f"     ❌ unpack failed for {p['filename']}: {e}")

print("\nDone.")

---

## 7. What's on disk now

Quick inventory of what we ended up with.

In [ ]:
def show_tree(root, max_entries: int = 30) -> None:
    from pathlib import Path
    root = Path(root)
    if not root.exists():
        print(f"  (no such dir: {root})")
        return
    entries = sorted(root.rglob("*"))
    print(f"  {root.relative_to(PROJECT_ROOT)}  ({len(entries)} entries)")
    for e in entries[:max_entries]:
        rel  = e.relative_to(root)
        size = f" {e.stat().st_size / 1e6:.1f} MB" if e.is_file() else ""
        kind = "📁" if e.is_dir() else "📄"
        print(f"    {kind} {rel}{size}")
    if len(entries) > max_entries:
        print(f"    ... and {len(entries) - max_entries} more")

for ds_name in target_datasets:
    print(f"\n=== {ds_name} ===")
    print("data/raw:")
    show_tree(PROJECT_ROOT / "data" / "raw" / ds_name)
    print("data/unpacked:")
    show_tree(PROJECT_ROOT / "data" / "unpacked" / ds_name)

---

## Next

- **ROG** → `11a_prep_ROG-art.ipynb`
- **ROG-Dialog** → `11b_prep_ROG.ipynb`
- **ParlaSpeech-{HR,RS,PL,CZ}** → `11c_prep_parlaspeech.ipynb` (set `cfg.lang` to match)
- **GOS** → arrange restricted audio access, then `prep_GOS.ipynb`

Re-running with the same config just prints skip lines — idempotent.